In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root, len(files))
    for f in files[:10]:
        print("   ", f)

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import random

DATA_DIR = Path(
    "/kaggle/input/datasets/bhanuprakashvecha/"
    "deeplure-saree-dataset/handloom_sarees"
)

images = list(DATA_DIR.glob("*.jpg"))

print("Total images:", len(images))

# Show 12 random images
sample_images = random.sample(images, min(12, len(images)))

plt.figure(figsize=(14, 10))

for i, image_path in enumerate(sample_images):
    img = Image.open(image_path).convert("RGB")

    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    plt.title(image_path.name)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np

DATA_DIR = Path(
    "/kaggle/input/datasets/bhanuprakashvecha/"
    "deeplure-saree-dataset/handloom_sarees"
)

images = list(DATA_DIR.glob("*.jpg"))

print("Total images:", len(images))

# Check image sizes
sizes = {}

for image_path in images:
    with Image.open(image_path) as img:
        size = img.size
        sizes[size] = sizes.get(size, 0) + 1

print("\nImage sizes:")
for size, count in sorted(sizes.items(), key=lambda x: -x[1]):
    print(size, ":", count)

# Check corrupted/unreadable images
bad_images = []

for image_path in images:
    try:
        with Image.open(image_path) as img:
            img.verify()
    except Exception:
        bad_images.append(str(image_path))

print("\nCorrupted images:", len(bad_images))

if bad_images:
    print(bad_images)

In [ ]:
from pathlib import Path
import hashlib

DATA_DIR = Path(
    "/kaggle/input/datasets/bhanuprakashvecha/"
    "deeplure-saree-dataset/handloom_sarees"
)

images = list(DATA_DIR.glob("*.jpg"))

hashes = {}
duplicates = []

for image_path in images:
    with open(image_path, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()

    if file_hash in hashes:
        duplicates.append(
            (image_path.name, hashes[file_hash])
        )
    else:
        hashes[file_hash] = image_path.name

print("Total images:", len(images))
print("Unique images:", len(hashes))
print("Exact duplicate images:", len(duplicates))

if duplicates:
    print("\nDuplicates:")
    for duplicate, original in duplicates:
        print(duplicate, "<--", original)

In [ ]:
from PIL import Image
from pathlib import Path

DATA_DIR = Path(
    "/kaggle/input/datasets/bhanuprakashvecha/"
    "deeplure-saree-dataset/handloom_sarees"
)

images = list(DATA_DIR.glob("*.jpg"))

print("Checking metadata of first 10 images...\n")

for image_path in images[:10]:

    img = Image.open(image_path)

    print("=" * 50)
    print("Filename:", image_path.name)
    print("Format:", img.format)
    print("Size:", img.size)
    print("Mode:", img.mode)

    if img.getexif():
        print("EXIF metadata found:", len(img.getexif()))
    else:
        print("EXIF metadata: None")

In [ ]:
import torch
import torchvision
from torchvision import models

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model.fc = torch.nn.Identity()

model = model.to(device)
model.eval()

print("ResNet18 loaded successfully.")

In [ ]:
from pathlib import Path
from PIL import Image
import torch
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np

DATA_DIR = Path(
    "/kaggle/input/datasets/bhanuprakashvecha/"
    "deeplure-saree-dataset/handloom_sarees"
)

images = sorted(DATA_DIR.glob("*.jpg"))

# Preprocessing required by pretrained ResNet18
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Generate embeddings
embeddings = []

model.eval()

with torch.no_grad():

    for image_path in images:

        img = Image.open(image_path).convert("RGB")
        img = transform(img).unsqueeze(0).to(device)

        feature = model(img)

        # Normalize embedding
        feature = torch.nn.functional.normalize(
            feature,
            p=2,
            dim=1
        )

        embeddings.append(feature.cpu())

embeddings = torch.cat(embeddings)

print("Number of images:", len(images))
print("Embedding shape:", embeddings.shape)

In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image

# Select a few query images
query_indices = [0, 20, 40, 60]

for query_idx in query_indices:

    query_embedding = embeddings[query_idx]

    # Cosine similarity because embeddings are normalized
    similarities = torch.matmul(
        embeddings,
        query_embedding
    )

    # Highest similarity first
    ranked_indices = torch.argsort(
        similarities,
        descending=True
    )

    # Remove the query image itself
    ranked_indices = [
        idx.item()
        for idx in ranked_indices
        if idx.item() != query_idx
    ]

    top_indices = ranked_indices[:5]

    # Display query + top 5
    plt.figure(figsize=(15, 3))

    # Query
    plt.subplot(1, 6, 1)

    img = Image.open(images[query_idx]).convert("RGB")

    plt.imshow(img)
    plt.title("QUERY")
    plt.axis("off")

    # Similar images
    for position, idx in enumerate(top_indices):

        plt.subplot(1, 6, position + 2)

        img = Image.open(images[idx]).convert("RGB")

        plt.imshow(img)

        score = similarities[idx].item()

        plt.title(
            f"Top {position + 1}\n{score:.3f}"
        )

        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
from torchvision import transforms
import torch

gray_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

gray_embeddings = []

model.eval()

with torch.no_grad():

    for image_path in images:

        img = Image.open(image_path).convert("RGB")

        img = gray_transform(img)

        img = img.unsqueeze(0).to(device)

        feature = model(img)

        feature = torch.nn.functional.normalize(
            feature,
            p=2,
            dim=1
        )

        gray_embeddings.append(
            feature.cpu()
        )

gray_embeddings = torch.cat(gray_embeddings)

print("Number of images:", len(images))
print("Grayscale embedding shape:", gray_embeddings.shape)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Use the same query images as Step 10
query_indices = [0, 20, 40, 60]

for query_idx in query_indices:

    # RGB similarities
    rgb_scores = torch.matmul(
        embeddings,
        embeddings[query_idx]
    )

    # Grayscale similarities
    gray_scores = torch.matmul(
        gray_embeddings,
        gray_embeddings[query_idx]
    )

    # Remove the query itself
    rgb_ranked = torch.argsort(
        rgb_scores, descending=True
    )

    gray_ranked = torch.argsort(
        gray_scores, descending=True
    )

    rgb_ranked = [
        i.item() for i in rgb_ranked
        if i.item() != query_idx
    ][:3]

    gray_ranked = [
        i.item() for i in gray_ranked
        if i.item() != query_idx
    ][:3]

    # Display results
    plt.figure(figsize=(14, 6))

    # Query
    plt.subplot(2, 4, 1)
    query_img = Image.open(
        images[query_idx]
    ).convert("RGB")
    plt.imshow(query_img)
    plt.title("QUERY")
    plt.axis("off")

    # RGB results
    for j, idx in enumerate(rgb_ranked):

        plt.subplot(2, 4, j + 2)

        img = Image.open(
            images[idx]
        ).convert("RGB")

        plt.imshow(img)

        plt.title(
            f"RGB Top {j+1}\n"
            f"{rgb_scores[idx]:.3f}"
        )

        plt.axis("off")

    # Grayscale label
    plt.subplot(2, 4, 5)
    plt.imshow(query_img)
    plt.title("GRAYSCALE RETRIEVAL")
    plt.axis("off")

    # Grayscale results
    for j, idx in enumerate(gray_ranked):

        plt.subplot(2, 4, j + 6)

        img = Image.open(
            images[idx]
        ).convert("RGB")

        plt.imshow(img)

        plt.title(
            f"Gray Top {j+1}\n"
            f"{gray_scores[idx]:.3f}"
        )

        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

color_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.8,
        hue=0.5
    )
])

# Pick 4 images
sample_indices = [0, 20, 40, 60]

plt.figure(figsize=(12, 10))

for row, idx in enumerate(sample_indices):

    original = Image.open(
        images[idx]
    ).convert("RGB")

    transformed = color_transform(original)

    # Original
    plt.subplot(4, 2, row * 2 + 1)
    plt.imshow(original)
    plt.title(f"Original: {images[idx].name}")
    plt.axis("off")

    # Color transformed
    plt.subplot(4, 2, row * 2 + 2)
    plt.imshow(transformed)
    plt.title("Color Augmented")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Create dataframe
df = pd.DataFrame({
    "image_path": [str(p) for p in images]
})

# Each original image gets a unique instance ID.
# This is NOT a real saree design label.
df["instance_id"] = [
    f"instance_{i:03d}"
    for i in range(len(df))
]

# Fixed seed for reproducibility
SEED = 42

# 70% train, 30% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED
)

# Split remaining 30% equally
# 15% validation, 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED
)

# Reset indexes
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Total images:", len(df))
print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

print("\nUnique train instances:", train_df["instance_id"].nunique())
print("Unique validation instances:", val_df["instance_id"].nunique())
print("Unique test instances:", test_df["instance_id"].nunique())

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch

# Training augmentation
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),

    transforms.RandomResizedCrop(
        size=224,
        scale=(0.75, 1.0)
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.8,
        hue=0.5
    ),

    transforms.RandomGrayscale(p=0.15),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Validation/test preprocessing
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class SareeInstanceDataset(Dataset):

    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        image_path = self.dataframe.loc[
            index, "image_path"
        ]

        image = Image.open(
            image_path
        ).convert("RGB")

        # Two independent augmented views
        view1 = self.transform(image)
        view2 = self.transform(image)

        return view1, view2


# Create training dataset
train_dataset = SareeInstanceDataset(
    train_df,
    train_transform
)

print("Training images:", len(train_dataset))

# Test one sample
view1, view2 = train_dataset[0]

print("View 1 shape:", view1.shape)
print("View 2 shape:", view2.shape)

In [ ]:
# Create DataLoader for training

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("Number of batches:", len(train_loader))

# Test one batch
batch_view1, batch_view2 = next(iter(train_loader))

print("Batch view 1 shape:", batch_view1.shape)
print("Batch view 2 shape:", batch_view2.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models


class SareeEmbeddingModel(nn.Module):

    def __init__(self, embedding_dim=128):

        super().__init__()

        # Pretrained ResNet18
        backbone = models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT
        )

        # ResNet18 produces 512-D features
        feature_dim = backbone.fc.in_features

        # Remove the original ImageNet classifier
        backbone.fc = nn.Identity()

        self.backbone = backbone

        # Convert 512-D features to 128-D embedding
        self.embedding = nn.Linear(
            feature_dim,
            embedding_dim
        )

    def forward(self, x):

        # Extract visual features
        features = self.backbone(x)

        # Create compact embedding
        embedding = self.embedding(features)

        # L2 normalize
        embedding = F.normalize(
            embedding,
            p=2,
            dim=1
        )

        return embedding


# Create model
embedding_model = SareeEmbeddingModel(
    embedding_dim=128
)

# Move model to GPU
embedding_model = embedding_model.to(device)

print(embedding_model)

In [ ]:
# Take one batch from the training DataLoader
batch_view1, batch_view2 = next(iter(train_loader))

# Move the batch to GPU
batch_view1 = batch_view1.to(device)
batch_view2 = batch_view2.to(device)

# Generate embeddings
with torch.no_grad():

    emb1 = embedding_model(batch_view1)
    emb2 = embedding_model(batch_view2)

print("Input shape:", batch_view1.shape)
print("Embedding 1 shape:", emb1.shape)
print("Embedding 2 shape:", emb2.shape)

# Check embedding normalization
norms = torch.norm(emb1, p=2, dim=1)

print("First 5 embedding norms:")
print(norms[:5])

In [ ]:
import torch
import torch.nn.functional as F


def contrastive_loss(z1, z2, temperature=0.07):

    # Normalize embeddings
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    batch_size = z1.size(0)

    # Combine both views
    z = torch.cat([z1, z2], dim=0)

    # Cosine similarity matrix
    similarity = torch.matmul(z, z.T) / temperature

    # Remove self-similarity from consideration
    mask = torch.eye(
        2 * batch_size,
        device=z.device,
        dtype=torch.bool
    )

    similarity.masked_fill_(mask, -1e9)

    # Positive pair for each sample
    labels = torch.arange(
        batch_size,
        device=z.device
    )

    labels = torch.cat([
        labels + batch_size,
        labels
    ])

    # Cross-entropy loss
    loss = F.cross_entropy(
        similarity,
        labels
    )

    return loss


# Test the loss using the embeddings
loss = contrastive_loss(
    emb1,
    emb2
)

print("Contrastive loss:", loss.item())

In [ ]:
# Optimizer with different learning rates
# for pretrained backbone and new embedding layer

optimizer = torch.optim.AdamW(
    [
        {
            "params": embedding_model.backbone.parameters(),
            "lr": 1e-5
        },
        {
            "params": embedding_model.embedding.parameters(),
            "lr": 1e-3
        }
    ],
    weight_decay=1e-4
)

print("Optimizer:", optimizer.__class__.__name__)

for i, group in enumerate(optimizer.param_groups):
    print(
        f"Parameter group {i + 1}: "
        f"learning rate = {group['lr']}, "
        f"weight decay = {group['weight_decay']}"
    )

In [ ]:
# Training configuration

NUM_EPOCHS = 15

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6
)

print("Number of epochs:", NUM_EPOCHS)
print("Scheduler:", scheduler.__class__.__name__)
print("Initial learning rates:")

for i, group in enumerate(optimizer.param_groups):
    print(
        f"Group {i + 1}: {group['lr']}"
    )

In [ ]:
# Training loop

training_losses = []

for epoch in range(NUM_EPOCHS):

    embedding_model.train()

    running_loss = 0.0

    for view1, view2 in train_loader:

        # Move images to GPU
        view1 = view1.to(device, non_blocking=True)
        view2 = view2.to(device, non_blocking=True)

        # Clear previous gradients
        optimizer.zero_grad()

        # Generate embeddings
        z1 = embedding_model(view1)
        z2 = embedding_model(view2)

        # Calculate contrastive loss
        loss = contrastive_loss(z1, z2)

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        running_loss += loss.item()

    # Average loss for this epoch
    epoch_loss = running_loss / len(train_loader)

    training_losses.append(epoch_loss)

    # Update learning rate
    scheduler.step()

    current_lr_backbone = optimizer.param_groups[0]["lr"]
    current_lr_embedding = optimizer.param_groups[1]["lr"]

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
        f"Loss: {epoch_loss:.4f} "
        f"| Backbone LR: {current_lr_backbone:.7f} "
        f"| Embedding LR: {current_lr_embedding:.7f}"
    )

print("\nTraining completed.")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(
    range(1, NUM_EPOCHS + 1),
    training_losses,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Contrastive Loss")
plt.title("Training Loss")

plt.xticks(range(1, NUM_EPOCHS + 1))
plt.grid(True)

plt.show()

print(
    "Lowest training loss:",
    min(training_losses)
)

print(
    "Best epoch:",
    training_losses.index(
        min(training_losses)
    ) + 1
)

In [ ]:
# Validation dataset
# We use the same augmentation pipeline to test
# color/appearance invariance.

val_dataset = SareeInstanceDataset(
    val_df,
    train_transform
)

print("Validation images:", len(val_dataset))

# Test one validation sample
val_view1, val_view2 = val_dataset[0]

print("Validation view 1 shape:", val_view1.shape)
print("Validation view 2 shape:", val_view2.shape)

In [ ]:
# Create validation DataLoader

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Number of validation batches:", len(val_loader))


# Calculate validation contrastive loss

embedding_model.eval()

validation_loss = 0.0

with torch.no_grad():

    for view1, view2 in val_loader:

        view1 = view1.to(
            device,
            non_blocking=True
        )

        view2 = view2.to(
            device,
            non_blocking=True
        )

        # Generate embeddings
        z1 = embedding_model(view1)
        z2 = embedding_model(view2)

        # Calculate loss
        loss = contrastive_loss(
            z1,
            z2
        )

        validation_loss += loss.item()


validation_loss /= len(val_loader)

print(
    "Validation contrastive loss:",
    validation_loss
)

In [ ]:
# Test dataset with deterministic preprocessing

test_dataset = SareeInstanceDataset(
    test_df,
    eval_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Test images:", len(test_dataset))
print("Number of test batches:", len(test_loader))

In [ ]:
# Generate deterministic embeddings for the test images

embedding_model.eval()

test_embeddings = []
test_paths = []

with torch.no_grad():

    for _, row in test_df.iterrows():

        image_path = row["image_path"]

        image = Image.open(
            image_path
        ).convert("RGB")

        image = eval_transform(image)

        image = image.unsqueeze(0).to(device)

        embedding = embedding_model(image)

        test_embeddings.append(
            embedding.cpu()
        )

        test_paths.append(image_path)


test_embeddings = torch.cat(
    test_embeddings,
    dim=0
)

print("Number of test images:", len(test_paths))
print("Test embedding shape:", test_embeddings.shape)

# Verify normalization
test_norms = torch.norm(
    test_embeddings,
    p=2,
    dim=1
)

print("First 5 embedding norms:")
print(test_norms[:5])

In [ ]:
# Create a strong color transformation for test queries

query_transform = transforms.Compose([
    transforms.Resize((256, 256)),

    transforms.CenterCrop(224),

    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.9,
        hue=0.5
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Generate one transformed query for every test image

torch.manual_seed(42)

query_embeddings = []

embedding_model.eval()

with torch.no_grad():

    for image_path in test_paths:

        image = Image.open(
            image_path
        ).convert("RGB")

        transformed_image = query_transform(
            image
        )

        transformed_image = (
            transformed_image
            .unsqueeze(0)
            .to(device)
        )

        embedding = embedding_model(
            transformed_image
        )

        query_embeddings.append(
            embedding.cpu()
        )


query_embeddings = torch.cat(
    query_embeddings,
    dim=0
)

print(
    "Query embedding shape:",
    query_embeddings.shape
)

# Check normalization
query_norms = torch.norm(
    query_embeddings,
    p=2,
    dim=1
)

print("First 5 query embedding norms:")
print(query_norms[:5])

In [ ]:
# Calculate cosine similarity between
# transformed queries and original test gallery

similarity_matrix = torch.matmul(
    query_embeddings,
    test_embeddings.T
)

print("Similarity matrix shape:")
print(similarity_matrix.shape)


# Rank gallery images for every query
ranked_indices = torch.argsort(
    similarity_matrix,
    dim=1,
    descending=True
)


# Calculate Recall@1 and Recall@5

correct_top1 = 0
correct_top5 = 0

for query_idx in range(len(test_paths)):

    ranked = ranked_indices[query_idx]

    # Correct gallery image has the same index
    correct_index = query_idx

    if ranked[0].item() == correct_index:
        correct_top1 += 1

    if correct_index in ranked[:5].tolist():
        correct_top5 += 1


recall_at_1 = (
    correct_top1 / len(test_paths)
)

recall_at_5 = (
    correct_top5 / len(test_paths)
)


print(f"Recall@1: {recall_at_1:.4f}")
print(f"Recall@5: {recall_at_5:.4f}")

print(
    f"Recall@1: {recall_at_1 * 100:.2f}%"
)

print(
    f"Recall@5: {recall_at_5 * 100:.2f}%"
)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Show the actual color-transformed query
# and its Top-5 gallery results

for query_idx in range(5):

    original_image = Image.open(
        test_paths[query_idx]
    ).convert("RGB")

    # Create the same transformed query
    torch.manual_seed(42 + query_idx)

    transformed_image = query_transform(
        original_image
    )

    # Convert normalized tensor back for display
    display_query = transformed_image.clone()

    display_query = display_query * torch.tensor(
        [0.229, 0.224, 0.225]
    ).view(3, 1, 1)

    display_query = display_query + torch.tensor(
        [0.485, 0.456, 0.406]
    ).view(3, 1, 1)

    display_query = torch.clamp(
        display_query,
        0,
        1
    )

    ranked = ranked_indices[query_idx]

    plt.figure(figsize=(15, 3))

    # Actual transformed query
    plt.subplot(1, 6, 1)

    plt.imshow(
        display_query.permute(1, 2, 0)
    )

    plt.title("COLOR-CHANGED\nQUERY")
    plt.axis("off")

    # Top 5 gallery results
    for position in range(5):

        gallery_idx = ranked[position].item()

        gallery_image = Image.open(
            test_paths[gallery_idx]
        ).convert("RGB")

        plt.subplot(
            1,
            6,
            position + 2
        )

        plt.imshow(gallery_image)

        score = similarity_matrix[
            query_idx,
            gallery_idx
        ].item()

        if gallery_idx == query_idx:

            title = (
                f"Top {position + 1}\n"
                f"CORRECT\n"
                f"{score:.3f}"
            )

        else:

            title = (
                f"Top {position + 1}\n"
                f"{score:.3f}"
            )

        plt.title(title)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
from PIL import Image


def generate_verification_embeddings(dataframe, seed_offset=1000):
    
    original_embeddings = []
    transformed_embeddings = []

    embedding_model.eval()

    with torch.no_grad():

        for i, (_, row) in enumerate(dataframe.iterrows()):

            image_path = row["image_path"]

            image = Image.open(
                image_path
            ).convert("RGB")

            # Original image
            original = eval_transform(image)
            original = original.unsqueeze(0).to(device)

            # Deterministic color-transformed version
            torch.manual_seed(seed_offset + i)

            transformed = query_transform(image)
            transformed = transformed.unsqueeze(0).to(device)

            # Generate embeddings
            original_embedding = embedding_model(
                original
            )

            transformed_embedding = embedding_model(
                transformed
            )

            original_embeddings.append(
                original_embedding.cpu()
            )

            transformed_embeddings.append(
                transformed_embedding.cpu()
            )

    return (
        torch.cat(original_embeddings),
        torch.cat(transformed_embeddings)
    )


# -----------------------------------------
# 1. Validation embeddings
# -----------------------------------------

val_original_embeddings, val_transformed_embeddings = (
    generate_verification_embeddings(
        val_df,
        seed_offset=1000
    )
)


# -----------------------------------------
# 2. Create validation positive pairs
# -----------------------------------------

val_positive_scores = torch.sum(
    val_original_embeddings *
    val_transformed_embeddings,
    dim=1
).numpy()


# -----------------------------------------
# 3. Create validation negative pairs
# -----------------------------------------

# Pair each transformed image with the
# ORIGINAL image belonging to the next instance.

val_negative_scores = []

num_val = len(val_original_embeddings)

for i in range(num_val):

    negative_index = (i + 1) % num_val

    score = torch.sum(
        val_original_embeddings[negative_index] *
        val_transformed_embeddings[i]
    ).item()

    val_negative_scores.append(score)


val_negative_scores = np.array(
    val_negative_scores
)


# -----------------------------------------
# 4. Create labels and scores
# -----------------------------------------

val_scores = np.concatenate([
    val_positive_scores,
    val_negative_scores
])

val_labels = np.concatenate([
    np.ones(len(val_positive_scores)),
    np.zeros(len(val_negative_scores))
])


# -----------------------------------------
# 5. Find best threshold on validation set
# -----------------------------------------

thresholds = np.linspace(
    val_scores.min(),
    val_scores.max(),
    200
)

best_threshold = None
best_f1 = -1

for threshold in thresholds:

    predictions = (
        val_scores >= threshold
    ).astype(int)

    tp = np.sum(
        (predictions == 1) &
        (val_labels == 1)
    )

    fp = np.sum(
        (predictions == 1) &
        (val_labels == 0)
    )

    fn = np.sum(
        (predictions == 0) &
        (val_labels == 1)
    )

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    if f1 > best_f1:

        best_f1 = f1
        best_threshold = threshold


print("Validation positive pairs:", len(val_positive_scores))
print("Validation negative pairs:", len(val_negative_scores))

print(
    f"Best validation threshold: "
    f"{best_threshold:.4f}"
)

print(
    f"Best validation F1: "
    f"{best_f1:.4f}"
)

In [ ]:
# -----------------------------------------
# Test verification evaluation
# -----------------------------------------

# Generate test original and transformed embeddings

test_original_embeddings, test_transformed_embeddings = (
    generate_verification_embeddings(
        test_df,
        seed_offset=2000
    )
)


# -----------------------------------------
# Positive pairs
# -----------------------------------------

test_positive_scores = torch.sum(
    test_original_embeddings *
    test_transformed_embeddings,
    dim=1
).numpy()


# -----------------------------------------
# Negative pairs
# -----------------------------------------

test_negative_scores = []

num_test = len(test_original_embeddings)

for i in range(num_test):

    negative_index = (i + 1) % num_test

    score = torch.sum(
        test_original_embeddings[negative_index] *
        test_transformed_embeddings[i]
    ).item()

    test_negative_scores.append(score)


test_negative_scores = np.array(
    test_negative_scores
)


# -----------------------------------------
# Combine scores and labels
# -----------------------------------------

test_scores = np.concatenate([
    test_positive_scores,
    test_negative_scores
])

test_labels = np.concatenate([
    np.ones(len(test_positive_scores)),
    np.zeros(len(test_negative_scores))
])


# -----------------------------------------
# Apply the FROZEN validation threshold
# -----------------------------------------

test_predictions = (
    test_scores >= best_threshold
).astype(int)


# -----------------------------------------
# Calculate metrics
# -----------------------------------------

tp = np.sum(
    (test_predictions == 1) &
    (test_labels == 1)
)

tn = np.sum(
    (test_predictions == 0) &
    (test_labels == 0)
)

fp = np.sum(
    (test_predictions == 1) &
    (test_labels == 0)
)

fn = np.sum(
    (test_predictions == 0) &
    (test_labels == 1)
)


accuracy = (
    (tp + tn) /
    len(test_labels)
)

precision = (
    tp / (tp + fp)
    if (tp + fp) > 0
    else 0
)

recall = (
    tp / (tp + fn)
    if (tp + fn) > 0
    else 0
)

f1 = (
    2 * precision * recall /
    (precision + recall)
    if (precision + recall) > 0
    else 0
)


print("Test positive pairs:", len(test_positive_scores))
print("Test negative pairs:", len(test_negative_scores))

print(
    f"\nFrozen threshold: {best_threshold:.4f}"
)

print(
    f"Test Accuracy: {accuracy:.4f} "
    f"({accuracy * 100:.2f}%)"
)

print(
    f"Test Precision: {precision:.4f} "
    f"({precision * 100:.2f}%)"
)

print(
    f"Test Recall: {recall:.4f} "
    f"({recall * 100:.2f}%)"
)

print(
    f"Test F1: {f1:.4f} "
    f"({f1 * 100:.2f}%)"
)

print("\nConfusion Matrix:")
print(
    f"TP={tp}, FP={fp}, "
    f"TN={tn}, FN={fn}"
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.hist(
    test_positive_scores,
    bins=10,
    alpha=0.7,
    label="Positive pairs"
)

plt.hist(
    test_negative_scores,
    bins=10,
    alpha=0.7,
    label="Negative pairs"
)

plt.axvline(
    best_threshold,
    linestyle="--",
    linewidth=2,
    label=f"Threshold = {best_threshold:.4f}"
)

plt.xlabel("Cosine Similarity")
plt.ylabel("Number of Pairs")
plt.title("Verification Score Distribution")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
import pandas as pd

# Final evaluation summary

results = pd.DataFrame({
    "Metric": [
        "Test Images",
        "Recall@1",
        "Recall@5",
        "Verification Threshold",
        "Verification Accuracy",
        "Verification Precision",
        "Verification Recall",
        "Verification F1",
        "True Positives",
        "False Positives",
        "True Negatives",
        "False Negatives"
    ],
    
    "Value": [
        len(test_paths),
        f"{recall_at_1 * 100:.2f}%",
        f"{recall_at_5 * 100:.2f}%",
        f"{best_threshold:.4f}",
        f"{accuracy * 100:.2f}%",
        f"{precision * 100:.2f}%",
        f"{recall * 100:.2f}%",
        f"{f1 * 100:.2f}%",
        tp,
        fp,
        tn,
        fn
    ]
})

display(results)

In [ ]:
import os
import torch

# Create output directory
OUTPUT_DIR = "/kaggle/working/deeplure_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save model checkpoint
model_path = os.path.join(
    OUTPUT_DIR,
    "saree_embedding_model.pth"
)

torch.save(
    {
        "model_state_dict": embedding_model.state_dict(),
        "embedding_dim": 128,
        "backbone": "ResNet18",
        "best_validation_threshold": float(best_threshold),
        "validation_f1": float(best_f1),
        "test_recall_at_1": float(recall_at_1),
        "test_recall_at_5": float(recall_at_5),
        "test_verification_accuracy": float(accuracy),
        "test_verification_precision": float(precision),
        "test_verification_recall": float(recall),
        "test_verification_f1": float(f1)
    },
    model_path
)

print("Model saved successfully.")
print("Model path:", model_path)

print(
    "File exists:",
    os.path.exists(model_path)
)

print(
    "File size (MB):",
    round(
        os.path.getsize(model_path) / (1024 * 1024),
        2
    )
)

In [ ]:
# Reload the saved model from disk

checkpoint = torch.load(
    model_path,
    map_location=device
)

# Create a fresh model
loaded_model = SareeEmbeddingModel(
    embedding_dim=checkpoint["embedding_dim"]
)

# Load saved weights
loaded_model.load_state_dict(
    checkpoint["model_state_dict"]
)

# Move to GPU and evaluation mode
loaded_model = loaded_model.to(device)
loaded_model.eval()

print("Saved model loaded successfully.")
print("Embedding dimension:", checkpoint["embedding_dim"])
print("Backbone:", checkpoint["backbone"])
print(
    "Saved validation threshold:",
    checkpoint["best_validation_threshold"]
)


# Verify it produces an embedding
sample_image = Image.open(
    test_paths[0]
).convert("RGB")

sample_tensor = eval_transform(
    sample_image
).unsqueeze(0).to(device)

with torch.no_grad():
    sample_embedding = loaded_model(
        sample_tensor
    )

print(
    "Loaded model embedding shape:",
    sample_embedding.shape
)

print(
    "Loaded model embedding norm:",
    torch.norm(
        sample_embedding,
        p=2,
        dim=1
    ).item()
)

In [ ]:
# Reusable saree design verification function

def get_embedding(image_path):
    
    image = Image.open(
        image_path
    ).convert("RGB")

    image = eval_transform(image)

    image = image.unsqueeze(0).to(device)

    with torch.no_grad():
        embedding = loaded_model(image)

    return embedding


def verify_saree_design(
    image_path_1,
    image_path_2,
    threshold=None
):

    if threshold is None:
        threshold = checkpoint[
            "best_validation_threshold"
        ]

    embedding_1 = get_embedding(
        image_path_1
    )

    embedding_2 = get_embedding(
        image_path_2
    )

    # Cosine similarity
    similarity = torch.sum(
        embedding_1 * embedding_2,
        dim=1
    ).item()

    result = (
        "MATCH"
        if similarity >= threshold
        else "NO MATCH"
    )

    return {
        "similarity": similarity,
        "threshold": threshold,
        "result": result
    }


# Test using two test images

result = verify_saree_design(
    test_paths[0],
    test_paths[1]
)

print("Image 1:", test_paths[0])
print("Image 2:", test_paths[1])
print(f"Similarity: {result['similarity']:.4f}")
print(f"Threshold: {result['threshold']:.4f}")
print(f"Result: {result['result']}")

In [ ]:
# Test a known positive pair
# Original image vs color-transformed version
# of the same image

positive_image_path = test_paths[0]

original_image = Image.open(
    positive_image_path
).convert("RGB")

# Original embedding
original_tensor = eval_transform(
    original_image
).unsqueeze(0).to(device)

# Create color-transformed version
torch.manual_seed(42)

transformed_tensor = query_transform(
    original_image
).unsqueeze(0).to(device)

# Generate embeddings
loaded_model.eval()

with torch.no_grad():

    original_embedding = loaded_model(
        original_tensor
    )

    transformed_embedding = loaded_model(
        transformed_tensor
    )

# Cosine similarity
positive_similarity = torch.sum(
    original_embedding *
    transformed_embedding,
    dim=1
).item()

threshold = checkpoint[
    "best_validation_threshold"
]

positive_result = (
    "MATCH"
    if positive_similarity >= threshold
    else "NO MATCH"
)

print("Original image:")
print(positive_image_path)

print(
    f"\nSimilarity: {positive_similarity:.4f}"
)

print(
    f"Threshold: {threshold:.4f}"
)

print(
    f"Result: {positive_result}"
)

In [ ]:
# ==========================================
# End-to-End Color-Invariant Saree Matching
# ==========================================

def run_color_invariant_demo(image_path):
    
    image = Image.open(
        image_path
    ).convert("RGB")

    # Original image
    original_tensor = eval_transform(
        image
    ).unsqueeze(0).to(device)

    # Create color-shifted query
    torch.manual_seed(42)

    query_tensor = query_transform(
        image
    ).unsqueeze(0).to(device)

    # Generate embeddings
    loaded_model.eval()

    with torch.no_grad():

        original_embedding = loaded_model(
            original_tensor
        )

        query_embedding = loaded_model(
            query_tensor
        )

    # Cosine similarity
    similarity = torch.sum(
        original_embedding *
        query_embedding,
        dim=1
    ).item()

    threshold = checkpoint[
        "best_validation_threshold"
    ]

    result = (
        "MATCH"
        if similarity >= threshold
        else "NO MATCH"
    )

    # Display images
    query_display = query_tensor.squeeze(0).cpu()

    query_display = query_display * torch.tensor(
        [0.229, 0.224, 0.225]
    ).view(3, 1, 1)

    query_display = query_display + torch.tensor(
        [0.485, 0.456, 0.406]
    ).view(3, 1, 1)

    query_display = torch.clamp(
        query_display,
        0,
        1
    )

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title("Original Saree")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(
        query_display.permute(1, 2, 0)
    )
    plt.title("Color-Transformed Query")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    print("Similarity:", f"{similarity:.4f}")
    print("Threshold:", f"{threshold:.4f}")
    print("Decision:", result)


# Run the complete demonstration
run_color_invariant_demo(
    test_paths[0]
)

# DeepLure – Color-Invariant Saree Design Recognition

## 1. Approach Note

We use a pretrained ResNet18 backbone followed by a 128-D L2-normalized embedding head. Contrastive learning with NT-Xent loss trains two augmented views of the same image to have similar embeddings while separating different instances. Strong color, crop, flip and grayscale augmentations encourage color-invariant representations. Cosine similarity is used for retrieval and verification.

## 2. Pipeline

RGB saree image
→ resize/crop
→ color and appearance augmentation
→ ResNet18 pretrained backbone
→ 512-D visual features
→ 128-D embedding
→ L2 normalization
→ cosine similarity
→ identification / verification

## 3. Training Strategy

- Backbone: pretrained ResNet18
- Embedding dimension: 128
- Loss: NT-Xent / InfoNCE contrastive loss
- Optimizer: AdamW
- Backbone learning rate: 1e-5
- Embedding-head learning rate: 1e-3
- Weight decay: 1e-4
- Scheduler: CosineAnnealingLR
- Epochs: 15
- Batch size: 16
- Augmentations: random resized crop, horizontal flip, ColorJitter and occasional grayscale

## 4. Dataset Split

The corpus contained 165 usable images.

- Training: 115 images
- Validation: 25 images
- Test: 25 images
- Random seed: 42

The split was performed before augmentation to prevent augmentation leakage between splits.

## 5. Identification Evaluation

For the controlled color-invariance retrieval experiment, each test image was used as a gallery item and a strongly color-transformed version was used as the query.

Results:

- Recall@1: 80.00%
- Recall@5: 100.00%

## 6. Verification Evaluation

A similarity threshold was selected using validation pairs only.

Validation threshold:

0.3721

The frozen threshold was then evaluated on unseen test pairs.

Results:

- Accuracy: 98.00%
- Precision: 96.15%
- Recall: 100.00%
- F1: 98.04%
- TP: 25
- FP: 1
- TN: 24
- FN: 0

## 7. Dataset Limitation

The supplied corpus does not contain explicit design IDs or colorway labels. Therefore, the positive pairs used in this experiment are generated by applying controlled color transformations to the same source image. This evaluates color-invariant embedding consistency, but it is not equivalent to measuring recognition between independently photographed real-world colorways.

Consequently, the reported Recall@K and verification metrics should be interpreted as controlled synthetic color-invariance results rather than definitive real-world design-recognition accuracy.

## 8. Reproducibility

- Framework: PyTorch
- Backbone: torchvision ResNet18 pretrained on ImageNet
- Embedding size: 128 dimensions
- Device: CUDA GPU
- Saved model:
  `/kaggle/working/deeplure_model/saree_embedding_model.pth`

The saved checkpoint contains the model weights, embedding dimension, validation threshold and evaluation metrics.